# LG HelloDoctor — B팀 의료 LLM 파인튜닝
> 데이터 로드 → LoRA 파인튜닝 → 의도분류 → Entity추출
> → Groq fallback → 평가 → 다중 턴 대화

## Step 1 — 라이브러리 설치

In [ ]:
!pip install unsloth trl datasets bitsandbytes groq -q
print('설치 완료!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2

## Step 2 — Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor/LLM/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor/LLM/model', exist_ok=True)
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf', exist_ok=True)
print('Drive 연결 완료!')

Mounted at /content/drive
Drive 연결 완료!


## Step 3 — 데이터 로드

In [ ]:

import gdown

gdown.download(
    'https://drive.google.com/uc?id=12QMAcn63E503HUMpVwiftj9ClycmkNvJ',
    '/content/00_all_medical_train.jsonl',
    quiet=False
)

Downloading...
From: https://drive.google.com/uc?id=12QMAcn63E503HUMpVwiftj9ClycmkNvJ
To: /content/00_all_medical_train.jsonl
100%|██████████| 477k/477k [00:00<00:00, 9.76MB/s]


'/content/00_all_medical_train.jsonl'

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files='/content/00_all_medical_train.jsonl', split='train')

print(f'데이터 로드 완료: {len(dataset)}개')
print('샘플 확인:')
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

데이터 로드 완료: 2500개
샘플 확인:
{'instruction': '있잖아요, 근데 저기요, 기침이 심해요', 'input': '', 'output': '많이 힘드시겠어요. 내과에 가보시겠어요?'}


In [ ]:
# from datasets import load_dataset

# dataset = load_dataset(
#     'json',
#     data_files='/content/drive/MyDrive/LG_HelloDoctor/LLM/data/00_all_medical_train.jsonl',
#     split='train'
# )
# print(result.stdout)
# print(f'데이터 로드 완료: {len(dataset)}개')
# print('샘플 확인:')
# print(dataset[0])


## Step 4 — 모델 로드 (4bit 양자화)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/llama-3.2-3b-instruct',
    max_seq_length=512,
    load_in_4bit=True,
)
print('모델 로드 완료!')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


모델 로드 완료!


## Step 5 — LoRA 설정

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing=True,
)
print('LoRA 설정 완료!')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.1 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA 설정 완료!


## Step 6 — 프롬프트 포맷

In [ ]:
def format_prompt(example):
    return {
        'text': f"""### 질문:\n{example['instruction']}\n\n### 답변:\n{example['output']}<|end_of_text|>"""
    }

dataset = dataset.map(format_prompt)
print('프롬프트 포맷 완료!')
print('샘플:', dataset[0]['text'][:100])

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

프롬프트 포맷 완료!
샘플: ### 질문:
있잖아요, 근데 저기요, 기침이 심해요

### 답변:
많이 힘드시겠어요. 내과에 가보시겠어요?<|end_of_text|>


## Step 7 — 학습

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    args=TrainingArguments(
        output_dir='/content/drive/MyDrive/LG_HelloDoctor/LLM/checkpoints',
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        save_total_limit=3,
        resume_from_checkpoint=True,
    ),
)

trainer.train()
print('학습 완료!')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,500 | Num Epochs = 3 | Total steps = 471
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.666010
20,1.428654
30,0.953129
40,0.703574
50,0.562430
60,0.488771
70,0.458789
80,0.394033
90,0.409442
100,0.357857


학습 완료!


## Step 8 — 모델 저장

In [ ]:
model.save_pretrained('/content/drive/MyDrive/LG_HelloDoctor/LLM/model')
tokenizer.save_pretrained('/content/drive/MyDrive/LG_HelloDoctor/LLM/model')
print('모델 저장 완료!')

모델 저장 완료!


## Step 9 — GGUF 변환

In [ ]:
model.save_pretrained_gguf(
    '/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf',
    tokenizer,
    quantization_method='q4_k_m'
)
print('GGUF 변환 완료!')

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:21<00:00, 40.62s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:46<00:00, 83.25s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/Modelfile
GGUF 변환 완료!


In [16]:
# 1. 설치
!pip install unsloth groq -q

# 2. Drive 연결
from google.colab import drive
drive.mount('/content/drive')

# 3. 저장된 모델 불러오기
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='/content/drive/MyDrive/LG_HelloDoctor/LLM/model',
    max_seq_length=512,
    load_in_4bit=True,
)
print('모델 불러오기 완료!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


모델 불러오기 완료!


## Step 10 — 의도 분류기

In [17]:
import re

# 응급 키워드 (LLM 거치지 않고 즉시 판단)
EMERGENCY_KEYWORDS = [
    '숨이 안 쉬어', '가슴이 너무 아프', '의식이 없',
    '쓰러', '피를 토', '말이 안 나', '한쪽이 마비',
    '갑자기 안 보여', '갑자기 못 움직', '말이 어눌',
    '입이 돌아', '숨을 못 쉬', '119'
]

INTENT_PROMPT = """다음 문장의 의도를 아래 4가지 중 하나로만 답하세요.
의도 종류:
- symptom_inquiry: 증상 문의 또는 진료과 질문
- hospital_search: 병원 위치, 운영시간 문의
- medication_info: 약 복용, 약 정보 문의
- emergency: 응급 상황

문장: {text}
의도:"""

def classify_intent(text: str) -> dict:
    # 1순위: 응급 키워드 즉시 판단
    for kw in EMERGENCY_KEYWORDS:
        if kw in text:
            return {'intent': 'emergency', 'confidence': 0.99, 'method': 'keyword'}

    # 2순위: 파인튜닝 모델로 분류
    prompt = INTENT_PROMPT.format(text=text)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=256
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=20,
        temperature=0.1,
        do_sample=True,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = result.split('의도:')[-1].strip().split()[0]

    valid_intents = ['symptom_inquiry', 'hospital_search', 'medication_info', 'emergency']
    intent = answer if answer in valid_intents else 'symptom_inquiry'

    return {'intent': intent, 'confidence': 0.85, 'method': 'llm'}


# 테스트
test_cases = [
    '무릎이 너무 아파요',
    '가슴이 너무 아프고 숨이 안 쉬어져요',
    '혈압약이랑 감기약 같이 먹어도 되나요?',
    '가까운 내과 어디예요?',
]

print('=== 의도 분류 테스트 ===')
for text in test_cases:
    result = classify_intent(text)
    print(f'입력: {text}')
    print(f'결과: {result}')
    print('-' * 40)

=== 의도 분류 테스트 ===


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

입력: 무릎이 너무 아파요
결과: {'intent': 'symptom_inquiry', 'confidence': 0.85, 'method': 'llm'}
----------------------------------------
입력: 가슴이 너무 아프고 숨이 안 쉬어져요
결과: {'intent': 'emergency', 'confidence': 0.99, 'method': 'keyword'}
----------------------------------------


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 혈압약이랑 감기약 같이 먹어도 되나요?
결과: {'intent': 'symptom_inquiry', 'confidence': 0.85, 'method': 'llm'}
----------------------------------------
입력: 가까운 내과 어디예요?
결과: {'intent': 'symptom_inquiry', 'confidence': 0.85, 'method': 'llm'}
----------------------------------------


## Step 11 — Entity 추출 (증상·부위·위치)

In [18]:
import json

BODY_PARTS = [
    '무릎', '허리', '어깨', '팔', '다리', '발', '손', '목',
    '머리', '눈', '귀', '코', '입', '치아', '잇몸', '가슴',
    '배', '위', '심장', '폐', '피부', '발목', '손목', '골반'
]

ENTITY_PROMPT = """다음 문장에서 증상과 신체 부위를 추출해서 JSON으로만 답하세요.
형식: {{"symptom": "증상", "body_part": "신체부위", "location": null}}
없으면 null로 표시하세요.

문장: {text}
JSON:"""

def extract_entities(text: str) -> dict:
    # 키워드 기반 빠른 추출
    found_parts = [p for p in BODY_PARTS if p in text]
    body_part = found_parts[0] if found_parts else None

    # LLM으로 정확한 추출
    prompt = ENTITY_PROMPT.format(text=text)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=256
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=60,
        temperature=0.1,
        do_sample=True,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_str = result.split('JSON:')[-1].strip()

    try:
        json_match = re.search(r'\{.*?\}', json_str, re.DOTALL)
        entities = json.loads(json_match.group()) if json_match else None
    except:
        entities = None

    if not entities:
        entities = {'symptom': None, 'body_part': body_part, 'location': None}

    return entities


# 테스트
test_cases = [
    '무릎이 너무 아파요. 어디 가야 해요?',
    '허리가 끊어질 것 같아요',
    '혈압약이랑 감기약 같이 먹어도 되나요?',
]

print('=== Entity 추출 테스트 ===')
for text in test_cases:
    entities = extract_entities(text)
    print(f'입력: {text}')
    print(f'결과: {entities}')
    print('-' * 40)

Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Entity 추출 테스트 ===


Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 무릎이 너무 아파요. 어디 가야 해요?
결과: {'symptom': '무릎이 너무 아파요', 'body_part': '무릎', 'location': None}
----------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 허리가 끊어질 것 같아요
결과: {'symptom': '허리가 끊어질 것 같아요', 'body_part': '허리', 'location': None}
----------------------------------------
입력: 혈압약이랑 감기약 같이 먹어도 되나요?
결과: {'symptom': None, 'body_part': None, 'location': None}
----------------------------------------


## Step 12 — Groq Fallback 연동

In [19]:
from groq import Groq
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

groq_client = Groq(api_key=GROQ_API_KEY)


SYSTEM_PROMPT = """당신은 노인 환자를 위한 의료 안내 AI 헬로비입니다.
반드시 아래 규칙을 지키세요:
- 3문장 이내로 답하세요
- 쉬운 말로 부드럽게 답하세요
- 존댓말을 사용하세요
- 질환을 단정하지 마세요
- 응급 상황이면 119를 먼저 안내하세요
- 금지 단어: 예후, 처방전, 투약, 병변"""

def generate_answer_local(text: str, context: str = '') -> dict:
    """파인튜닝 모델로 답변 생성"""
    prompt = f'### 질문:\n{text}\n\n### 답변:\n'
    if context:
        prompt = f'참고 정보: {context}\n\n{prompt}'

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = result.split('### 답변:')[-1].strip()
    return {'answer': answer, 'model': 'LG_HelloDoctor/LLM'}


def generate_answer_groq(text: str, context: str = '') -> dict:
    """Groq API로 답변 생성 (fallback)"""
    user_msg = f'참고 정보: {context}\n\n질문: {text}' if context else text

    response = groq_client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_msg}
        ],
        max_tokens=150,
        temperature=0.3,
    )
    answer = response.choices[0].message.content.strip()
    return {'answer': answer, 'model': 'groq-llama-3.3-70b'}


def generate_answer(text: str, context: str = '', confidence: float = 0.85) -> dict:
    """신뢰도 기반 라우팅"""
    if confidence >= 0.7:
        try:
            return generate_answer_local(text, context)
        except Exception as e:
            print(f'로컬 오류 → Groq fallback: {e}')
            return generate_answer_groq(text, context)
    else:
        print('신뢰도 낮음 → Groq fallback')
        return generate_answer_groq(text, context)


# 테스트
print('=== 로컬 모델 ===')
r = generate_answer('무릎이 너무 아파요. 어디 가야 해요?', confidence=0.91)
print(f'모델: {r["model"]}\n답변: {r["answer"]}')

print('\n=== Groq Fallback ===')
r = generate_answer('무릎이 너무 아파요. 어디 가야 해요?', confidence=0.5)
print(f'모델: {r["model"]}\n답변: {r["answer"]}')

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 로컬 모델 ===


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


모델: LG_HelloDoctor/LLM
답변: 많이 아프시겠어요. 정형외과에 가보시는 게 좋을 것 같아요.

=== Groq Fallback ===
신뢰도 낮음 → Groq fallback
모델: groq-llama-3.3-70b
답변: 무릎이 아프면 병원에 가서 의사 선생님께 진찰을 받는 것이 좋습니다. 근처에 있는 병원이나 정형외과를 방문해 보세요. 혹시 심한 통증이나 부상이 있다면 119에 연락하세요.


## Step 13 — 다중 턴 대화 (추가 질문)

In [20]:
# 대화 상태 저장소
conversation_state = {}

# 신체 부위별 추가 질문
FOLLOWUP_QUESTIONS = {
    '무릎': '무릎이 아프시군요. 걷기는 괜찮으세요?',
    '허리': '허리가 아프시군요. 허리를 펴기 힘드세요?',
    '어깨': '어깨가 아프시군요. 팔을 올리기 힘드세요?',
    '배': '배가 아프시군요. 속이 쓰리거나 울렁거리세요?',
    '머리': '머리가 아프시군요. 갑자기 생긴 통증인가요?',
    '가슴': '가슴이 아프시군요. 숨쉬기는 괜찮으세요?',
    '눈': '눈이 불편하시군요. 갑자기 안 보이시는 건가요?',
    '귀': '귀가 불편하시군요. 갑자기 안 들리시는 건가요?',
    '발': '발이 아프시군요. 걷기가 힘드세요?',
    '손': '손이 불편하시군요. 손을 쥐기 힘드세요?',
    '목': '목이 아프시군요. 침을 삼키기 힘드세요?',
    '피부': '피부가 불편하시군요. 많이 가렵거나 부어오르셨나요?',
}

# 심각도별 병원 안내
SEVERITY_DEPT_MAP = {
    '무릎': {
        '심함': '정형외과에 가보시는 게 좋을 것 같아요.',
        '가벼움': '정형외과에 가보시는 게 좋을 것 같아요. 가볍게 아프시면 내과도 괜찮아요.'
    },
    '허리': {
        '심함': '정형외과나 신경외과에 가보시는 게 좋을 것 같아요.',
        '가벼움': '정형외과에 가보시는 게 좋을 것 같아요.'
    },
    '어깨': {
        '심함': '정형외과에 가보시는 게 좋을 것 같아요.',
        '가벼움': '정형외과에 가보시는 게 좋을 것 같아요.'
    },
    '배': {
        '심함': '내과나 외과에 가보시는 게 좋을 것 같아요.',
        '가벼움': '소화기내과에 가보시는 게 좋을 것 같아요.'
    },
    '머리': {
        '심함': '신경과나 응급실에 가보시는 게 좋을 것 같아요.',
        '가벼움': '내과나 신경과에 가보시는 게 좋을 것 같아요.'
    },
    '가슴': {
        '심함': '119에 바로 전화해 주세요.',
        '가벼움': '심장내과나 내과에 가보시는 게 좋을 것 같아요.'
    },
    '눈': {
        '심함': '안과에 빨리 가보시는 게 좋을 것 같아요.',
        '가벼움': '안과에 가보시는 게 좋을 것 같아요.'
    },
    '귀': {
        '심함': '이비인후과에 빨리 가보시는 게 좋을 것 같아요.',
        '가벼움': '이비인후과에 가보시는 게 좋을 것 같아요.'
    },
    '발': {
        '심함': '정형외과에 가보시는 게 좋을 것 같아요.',
        '가벼움': '정형외과에 가보시는 게 좋을 것 같아요.'
    },
    '손': {
        '심함': '정형외과에 가보시는 게 좋을 것 같아요.',
        '가벼움': '정형외과에 가보시는 게 좋을 것 같아요.'
    },
    '목': {
        '심함': '이비인후과에 가보시는 게 좋을 것 같아요.',
        '가벼움': '이비인후과에 가보시는 게 좋을 것 같아요.'
    },
    '피부': {
        '심함': '피부과에 빨리 가보시는 게 좋을 것 같아요.',
        '가벼움': '피부과에 가보시는 게 좋을 것 같아요.'
    },
}

# 심각도 키워드
SEVERITY_KEYWORDS = {
    '심함': ['많이', '너무', '심하게', '못', '힘들', '안 돼', '못 걷', '못 들', '매우', '엄청', '심해'],
    '가벼움': ['조금', '약간', '가끔', '괜찮', '그냥', '좀', '살짝', '가볍게']
}

def detect_severity(text: str) -> str:
    for kw in SEVERITY_KEYWORDS['심함']:
        if kw in text:
            return '심함'
    for kw in SEVERITY_KEYWORDS['가벼움']:
        if kw in text:
            return '가벼움'
    return '심함'  # 기본값은 심함 (안전하게)


def chat_with_followup(user_input: str, session_id: str = 'default') -> str:
    """
    다중 턴 대화:
    1턴: 증상 파악 → 추가 질문
    2턴: 심각도 파악 → 병원 안내
    응급: 즉시 119 안내 (턴 무관)
    """
    global conversation_state

    # 세션 초기화
    if session_id not in conversation_state:
        conversation_state[session_id] = {
            'step': 1,
            'body_part': None
        }

    state = conversation_state[session_id]

    # 응급 먼저 체크 (턴 무관)
    intent = classify_intent(user_input)
    if intent['intent'] == 'emergency':
        conversation_state[session_id] = {'step': 1, 'body_part': None}
        return '지금 바로 119에 전화해 주세요. 매우 위험한 상황이에요.'

    # 복약·병원 검색은 바로 처리
    if intent['intent'] in ['medication_info', 'hospital_search']:
        result = generate_answer(user_input)
        return result['answer']

    # 1턴: 신체 부위 파악 → 추가 질문
    if state['step'] == 1:
        for part in FOLLOWUP_QUESTIONS.keys():
            if part in user_input:
                state['body_part'] = part
                state['step'] = 2
                return FOLLOWUP_QUESTIONS[part]
        # 부위 못 찾으면 LLM으로 바로 답변
        result = generate_answer(user_input)
        return result['answer']

    # 2턴: 심각도 파악 → 병원 안내
    if state['step'] == 2:
        body_part = state['body_part']
        severity = detect_severity(user_input)
        # 상태 초기화 (다음 대화를 위해)
        conversation_state[session_id] = {'step': 1, 'body_part': None}

        if body_part and body_part in SEVERITY_DEPT_MAP:
            return SEVERITY_DEPT_MAP[body_part][severity]
        else:
            result = generate_answer(user_input)
            return result['answer']


# 테스트
print('=== 다중 턴 대화 테스트 ===')
print('입력: 무릎이 아파요')
print(f'AI:   {chat_with_followup("무릎이 아파요", "test_1")}')
print('입력: 많이 힘들고 걷기 어려워요')
print(f'AI:   {chat_with_followup("많이 힘들고 걷기 어려워요", "test_1")}')
print()
print('입력: 허리가 아파요')
print(f'AI:   {chat_with_followup("허리가 아파요", "test_2")}')
print('입력: 조금 뻐근한 정도예요')
print(f'AI:   {chat_with_followup("조금 뻐근한 정도예요", "test_2")}')

=== 다중 턴 대화 테스트 ===
입력: 무릎이 아파요


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI:   무릎이 아프시군요. 걷기는 괜찮으세요?
입력: 많이 힘들고 걷기 어려워요


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI:   지금 바로 119에 전화해 주세요. 매우 위험한 상황이에요.

입력: 허리가 아파요


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI:   허리가 아프시군요. 허리를 펴기 힘드세요?
입력: 조금 뻐근한 정도예요
AI:   정형외과에 가보시는 게 좋을 것 같아요.


## Step 14 — 통합 테스트 (다중 턴 포함)

In [21]:
import json

def full_pipeline_v2(turns: list, session_id: str) -> None:
    """
    다중 턴 통합 테스트
    turns: [(사용자 발화, A팀 전달 형식), ...]
    """
    for i, (user_input, input_from_A) in enumerate(turns):
        print(f'{i+1}턴 사용자: {user_input}')
        answer = chat_with_followup(input_from_A['text'], session_id)
        print(f'{i+1}턴 AI:     {answer}')
    print()


# 시나리오 A — 증상 + 심한 경우
print('=' * 50)
print('시나리오 A: 무릎 통증 (심한 경우)')
print('=' * 50)
full_pipeline_v2([
    ('무릎이 아파요', {'text': '무릎이 아파요', 'raw_text': '헬로비 무릎이 아파요', 'confidence': 0.94, 'language': 'ko'}),
    ('많이 힘들고 걷기 어려워요', {'text': '많이 힘들고 걷기 어려워요', 'raw_text': '많이 힘들고 걷기 어려워요', 'confidence': 0.92, 'language': 'ko'}),
], 'scenario_A')

# 시나리오 B — 응급 (1턴 즉시 처리)
print('=' * 50)
print('시나리오 B: 응급 상황')
print('=' * 50)
full_pipeline_v2([
    ('가슴이 아프고 숨이 안 쉬어져요', {'text': '가슴이 아프고 숨이 안 쉬어져요', 'raw_text': '헬로비야 가슴이 아프고 숨이 안 쉬어져요', 'confidence': 0.97, 'language': 'ko'}),
], 'scenario_B')

# 시나리오 C — 복약 (1턴 즉시 처리)
print('=' * 50)
print('시나리오 C: 복약 문의')
print('=' * 50)
full_pipeline_v2([
    ('혈압약이랑 감기약 같이 먹어도 되나요?', {'text': '혈압약이랑 감기약 같이 먹어도 되나요', 'raw_text': '헬로비 혈압약이랑 감기약 같이 먹어도 되나요', 'confidence': 0.91, 'language': 'ko'}),
], 'scenario_C')

# 시나리오 D — 증상 + 가벼운 경우
print('=' * 50)
print('시나리오 D: 허리 통증 (가벼운 경우)')
print('=' * 50)
full_pipeline_v2([
    ('허리가 아파요', {'text': '허리가 아파요', 'raw_text': '헬로비 허리가 아파요', 'confidence': 0.93, 'language': 'ko'}),
    ('조금 뻐근한 정도예요', {'text': '조금 뻐근한 정도예요', 'raw_text': '조금 뻐근한 정도예요', 'confidence': 0.90, 'language': 'ko'}),
], 'scenario_D')

# 시나리오 E — 병원 검색 (1턴 즉시 처리)
print('=' * 50)
print('시나리오 E: 병원 검색')
print('=' * 50)
full_pipeline_v2([
    ('가까운 내과 알려주세요', {'text': '가까운 내과 알려주세요', 'raw_text': '헬로비 가까운 내과 알려주세요', 'confidence': 0.95, 'language': 'ko'}),
], 'scenario_E')

Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


시나리오 A: 무릎 통증 (심한 경우)
1턴 사용자: 무릎이 아파요


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1턴 AI:     무릎이 아프시군요. 걷기는 괜찮으세요?
2턴 사용자: 많이 힘들고 걷기 어려워요


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

1턴 AI:     허리가 아프시군요. 허리를 펴기 힘드세요?
2턴 사용자: 조금 뻐근한 정도예요


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2턴 AI:     정형외과에 가보시는 게 좋을 것 같아요.

시나리오 E: 병원 검색
1턴 사용자: 가까운 내과 알려주세요


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1턴 AI:     근처 내과에 가보시는 게 좋을 것 같아요.



## Step 15 — 모델 평가 및 성능 비교

In [22]:
import json

# 평가 데이터 로드 (1,000개)
eval_data = []
with open('/content/drive/MyDrive/LG_HelloDoctor/LLM/data/eval_00_all.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        eval_data.append((
            item['text'],
            item['intent'],
            item['department']
        ))

print(f'평가 데이터 로드 완료: {len(eval_data)}개')

def evaluate_model():
    intent_correct = 0
    dept_correct = 0
    dept_total = 0
    emergency_correct = 0
    emergency_total = 0
    korean_correct = 0
    wrong_cases = []

    for text, true_intent, true_dept in eval_data:
        # 의도 분류
        intent_result = classify_intent(text)
        pred_intent = intent_result['intent']

        # 답변 생성
        answer_result = generate_answer(text, confidence=intent_result['confidence'])
        answer = answer_result['answer']

        # 의도 분류 정확도
        intent_ok = pred_intent == true_intent
        if intent_ok:
            intent_correct += 1
        else:
            wrong_cases.append({'text': text, 'true': true_intent, 'pred': pred_intent})

        # 응급 감지율
        if true_intent == 'emergency':
            emergency_total += 1
            if pred_intent == 'emergency':
                emergency_correct += 1

        # 진료과 정확도
        if true_dept:
            dept_total += 1
            if true_dept in answer:
                dept_correct += 1

        # 한국어 응답률
        korean_ratio = sum(1 for c in answer if '가' <= c <= '힣') / max(len(answer), 1)
        if korean_ratio > 0.3:
            korean_correct += 1

    total = len(eval_data)
    print('=' * 50)
    print('📊 최종 모델 평가 결과 (1,000개)')
    print('=' * 50)
    print(f'의도 분류 정확도:  {intent_correct}/{total} = {intent_correct/total*100:.1f}%  (목표: 80%↑)')
    print(f'응급 감지율:       {emergency_correct}/{emergency_total} = {emergency_correct/emergency_total*100:.1f}%  (목표: 95%↑)')
    print(f'진료과 정확도:     {dept_correct}/{dept_total} = {dept_correct/dept_total*100:.1f}%  (목표: 80%↑)')
    print(f'한국어 응답률:     {korean_correct}/{total} = {korean_correct/total*100:.1f}%  (목표: 99%↑)')
    print('=' * 50)

    if wrong_cases:
        print(f'\n❌ 틀린 의도 분류 ({len(wrong_cases)}개 중 최대 5개):')
        for w in wrong_cases[:5]:
            print(f'  입력: {w["text"]}')
            print(f'  정답: {w["true"]} / 예측: {w["pred"]}')
            print()

evaluate_model()

Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


평가 데이터 로드 완료: 1000개


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_ge

📊 최종 모델 평가 결과 (1,000개)
의도 분류 정확도:  576/1000 = 57.6%  (목표: 80%↑)
응급 감지율:       114/200 = 57.0%  (목표: 95%↑)
진료과 정확도:     119/200 = 59.5%  (목표: 80%↑)
한국어 응답률:     1000/1000 = 100.0%  (목표: 99%↑)

❌ 틀린 의도 분류 (424개 중 최대 5개):
  입력: 방금 온몸에 갑자기 두드러기가 났어요
  정답: emergency / 예측: symptom_inquiry

  입력: 헬로비 헬로비 헬로비 오늘 진료하는 병원 있어요?
  정답: hospital_search / 예측: symptom_inquiry

  입력: 산부인과 근처에 있나요?
  정답: hospital_search / 예측: symptom_inquiry

  입력: 헬로비 있잖아요 이가 시려서 못 먹겠어요
  정답: symptom_inquiry / 예측: medication_info

  입력: 머리를 세게 부딪혔어요
  정답: emergency / 예측: symptom_inquiry



In [23]:
import os
os.chdir('/content')
!rm -rf LGHelloDoctor

# 토큰 포함해서 클론
!git clone https://YOUR_TOKEN_HERE@github.com/lg-hellovision-dx-data-school/LGHelloDoctor.git
os.chdir('/content/LGHelloDoctor')

# llm 브랜치로 이동
!git checkout llm

# 파일 복사
!cp /content/drive/MyDrive/LG_HelloDoctor/LLM/B_llm_full.ipynb .
!mkdir -p data
!cp /content/drive/MyDrive/LG_HelloDoctor/LLM/data/00_all_medical_train.jsonl data/
!cp /content/drive/MyDrive/LG_HelloDoctor/LLM/data/eval_00_all.jsonl data/

# 커밋 & 푸시
!git config user.email "dkswndus6988@naver.com"
!git config user.name "dkswndus"
!git add .
!git commit -m "feat: 다중 턴 대화 · 평가 코드 · 평가 데이터 추가"
!git push origin llm

Cloning into 'LGHelloDoctor'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 110 (delta 28), reused 95 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 274.68 KiB | 6.39 MiB/s, done.
Resolving deltas: 100% (28/28), done.
Branch 'llm' set up to track remote branch 'llm' from 'origin'.
Switched to a new branch 'llm'
[llm cb054c0] feat: 다중 턴 대화 · 평가 코드 · 평가 데이터 추가
 2 files changed, 1001 insertions(+), 1 deletion(-)
 rewrite B_llm_full.ipynb (93%)
 create mode 100644 data/eval_00_all.jsonl
remote: Permission to lg-hellovision-dx-data-school/LGHelloDoctor.git denied to dkswndus.
fatal: unable to access 'https://github.com/lg-hellovision-dx-data-school/LGHelloDoctor.git/': The requested URL returned error: 403


In [25]:
!git remote set-url origin https://YOUR_TOKEN_HERE@github.com/lg-hellovision-dx-data-school/LGHelloDoctor.git
!git push origin llm

remote: Permission to lg-hellovision-dx-data-school/LGHelloDoctor.git denied to dkswndus.
fatal: unable to access 'https://github.com/lg-hellovision-dx-data-school/LGHelloDoctor.git/': The requested URL returned error: 403


In [26]:
# 현재 remote URL 확인
!git remote -v

origin	https://YOUR_TOKEN_HERE@github.com/lg-hellovision-dx-data-school/LGHelloDoctor.git (fetch)
origin	https://YOUR_TOKEN_HERE@github.com/lg-hellovision-dx-data-school/LGHelloDoctor.git (push)
